# 웹에서 데이터를 꺼내는 법

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

# 🕸️ 분석한 데이터는 어디서 왔는가 — 웹에서 데이터 꺼내기

### — HTTP의 원리부터 BeautifulSoup 정적 수집, 그리고 AI 보조 코딩까지 —

---

## 📋 학습 목표

이 자습서를 마치면 여러분은 다음을 할 수 있습니다.

1. 웹이 데이터를 주고받는 방식(HTTP 요청·응답)과 정적·동적 페이지의 차이를 설명할 수 있습니다.
2. 스크래핑 도구 생태계(BeautifulSoup·Selenium·Playwright)의 용도 차이를 설명할 수 있습니다.
3. BeautifulSoup으로 정적 페이지에서 원하는 데이터를 골라 추출할 수 있습니다.
4. AI에게 스크래퍼 코드를 받아 **직접 검증하고 고쳐** 신뢰할 수 있는 코드로 만들 수 있습니다.
5. 수집한 데이터를 DataFrame으로 바꾸고, 지난 모듈의 랭글링 기술(결측·문자열 정제)을 다시 적용해 분석 가능한 표로 만들 수 있습니다.
6. 수집 대상 페이지의 유형을 판별하고 적합한 도구를 고를 수 있습니다.
7. 수집 전 robots.txt와 요청 예의를 확인하는 **책임 있는 수집 습관**을 들일 수 있습니다.

> 💡 위 목록을 천천히 읽고, 지금 내가 할 수 있는 것과 아직 낯선 것을 마음속으로 표시해보세요. 자습서를 다 마친 뒤 다시 돌아와 비교하면 여러분의 성장이 눈에 보일 거예요.

## 📚 목차

| Part | 내용 | 핵심 질문 |
| --- | --- | --- |
| Part 0 | 분석 상황과 학습 지도 | 우리가 분석하던 데이터는 대체 어디서 왔을까요? |
| Part 1 | 왜 수집인가 — 분석의 입구 | 데이터가 주어지지 않으면 어떻게 구할까요? |
| Part 2 | HTTP 요청과 응답 | 웹 브라우저와 서버는 어떻게 대화할까요? |
| Part 3 | 정적 vs 동적 페이지 | 화면엔 보이는데 소스엔 없는 데이터가 있다고요? |
| Part 4 | HTML과 DOM 트리 | 페이지의 뼈대는 어떻게 생겼을까요? |
| Part 5 | BeautifulSoup 정적 수집 | HTML에서 원하는 값만 어떻게 골라낼까요? |
| Part 6 | 수집 → DataFrame → 랭글링 | 꺼낸 데이터를 분석 가능한 표로 어떻게 만들까요? |
| Part 7 | 도구 지형 + AI 보조 코딩 | 어떤 도구를 고르고, AI를 어떻게 검증하며 쓸까요? |
| Part 8 | 책임 있는 수집 (윤리·법) | 수집해도 되는지 어떻게 판단할까요? |
| 종합 실습 | 모두마켓 상품 수집 파이프라인 | 수집부터 정제까지 혼자 해낼 수 있을까요? |
| 정리 | 핵심 요약 · 다음 시간 | 정적으로 안 되는 페이지는 어떻게 할까요? |

## 분석 상황과 학습 지도

# 0. 분석 상황과 학습 지도

## 지난 모듈에서는 무엇을 했나요?

지난 모듈(데이터 랭글링)에서 우리는 **지저분한 데이터를 분석 가능한 형태로 닦는** 일을 했습니다. 결측치를 처리하고, 이상치를 다루고, 문자열·날짜를 정리하고, 여러 테이블을 병합하고, 전처리 파이프라인까지 만들었죠. 모듈의 마지막 마무리에서 이런 질문을 남겼습니다.

> "그런데… 우리가 그렇게 열심히 정제한 이 데이터, **대체 어디서 온 걸까요?**"

지금까지는 데이터가 **이미 손에 주어진** 상태로 시작했습니다. 하지만 현장에서 데이터는 종종 **직접 구해 와야** 합니다. 그중에서도 가장 흔한 원천이 바로 **웹**입니다.

## 오늘의 여정

오늘은 "데이터의 출처"를 다룹니다. 웹이 데이터를 주고받는 원리부터, 실제로 웹 페이지에서 원하는 값을 꺼내 분석 가능한 표로 만드는 데까지 한 흐름으로 따라갑니다.

```text
[Part 1] 왜 수집인가        →  분석의 입구, 수집 방법의 지형
   ↓
[Part 2] HTTP 요청·응답     →  웹은 어떻게 대화하는가
   ↓
[Part 3] 정적 vs 동적        →  소스에 데이터가 있는가 없는가
   ↓
[Part 4] HTML / DOM 트리    →  페이지의 뼈대를 읽는 법
   ↓
[Part 5] BeautifulSoup      →  HTML에서 원하는 값만 골라내기
   ↓
[Part 6] 수집 → 정제         →  DataFrame 변환 + 랭글링 재적용
   ↓
[Part 7] 도구 + AI 보조      →  도구 선택, AI 코드 검증 워크플로
   ↓
[Part 8] 책임 있는 수집      →  robots.txt · 요청 예의 · 윤리
   ↓
[종합 실습] 모두마켓 상품 수집 파이프라인 → 수집부터 정제까지
```

## 이 자습서 사용법

이 노트북은 **혼자서도 끝까지 학습할 수 있도록** 만들었습니다. 다음 네 박자로 진행하세요.

- 📖 **읽고** — 개념 설명을 천천히 읽습니다.
- 💻 **실행하고** — 코드 셀을 위에서부터 순서대로 실행합니다. (단축키: `Shift + Enter`)
- ✏️ **고쳐보고** — "스스로 해보자!" 칸에서 코드를 직접 바꿔봅니다.
- 🤔 **답해보고** — 체크포인트 질문에 스스로 답해봅니다. 틀려도 괜찮습니다.

> ⚠ **이 노트북의 특별한 점 — 인터넷 없이도 됩니다:** 웹 스크래핑은 "살아 있는 웹"을 다루다 보니, 사이트가 느리거나 막히면 코드가 실패할 수 있습니다. 그래서 이 노트북의 **핵심 학습은 노트북 안에 넣어 둔 HTML**(인터넷 불필요)로 진행합니다. 인터넷이 필요한 셀에는 `# [네트워크 필요]` 주석을 달았고, 연결이 안 되면 **자동으로 로컬 예시로 대체**되어 같은 내용을 학습할 수 있습니다.

In [ ]:
# ─────────────────────────────────────────────
# 환경 준비 — 라이브러리 불러오기 + 설치/네트워크 점검
# ─────────────────────────────────────────────
# 처음이라면 아래 주석을 해제해 설치하세요. (bs4 = BeautifulSoup)
# !pip install requests beautifulsoup4 pandas -q

import platform
import warnings
import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")
np.random.seed(42)

# 핵심 도구 1: BeautifulSoup — HTML을 파싱(parsing, 구조 분석)하는 라이브러리
from bs4 import BeautifulSoup

# 핵심 도구 2: requests — 웹에 요청을 보내는 라이브러리 (없을 수도 있으니 안전하게 점검)
try:
    import requests
    HAS_REQUESTS = True
except Exception:
    HAS_REQUESTS = False

print("준비 완료!")
print("BeautifulSoup: 사용 가능 (핵심 학습은 이 도구로 진행)")
print("requests     :", "사용 가능" if HAS_REQUESTS else "없음 → 네트워크 셀은 로컬 폴백으로 대체됩니다")

## 왜 수집인가 — 분석의 입구

# 1. 왜 수집인가 — 분석의 입구

데이터 분석의 모든 단계는 "데이터가 있다"는 전제에서 출발합니다. 그런데 그 데이터는 누군가가 **어딘가에서 가져온 것**입니다. 회사 DB에서 뽑았거나, 공공데이터를 내려받았거나, 혹은 **웹에서 긁어왔거나(scraping)**. 분석가가 "이 데이터 좀 구해줄 수 있어요?"라는 요청을 받는 순간, 가장 먼저 결정하는 것이 **어떻게 수집할 것인가**입니다.

> ❓ **이 파트에서 답할 질문:** 데이터가 깔끔한 파일로 주어지지 않을 때, 우리는 어떻게 데이터를 구할까요?

## 💡 쉽게 말하면 — 수집은 분석 파이프라인의 '입구'

지난 모듈에서 배운 랭글링(정제)은 사실 **수집 다음 단계**입니다. 전체 흐름에서 오늘 배우는 수집이 어디에 있는지 봅시다.

```text
[수집]          →  [정제(랭글링)]   →  [분석/통계]   →  [전달]
 오늘 배움          지난 모듈            다음 모듈        보고서·대시보드
 웹·API·파일       결측·문자열·병합      검정·인과추정
   ▲
   └ 여기가 막히면 그 뒤가 전부 멈춥니다. 수집은 분석의 '입구'입니다.
```

수집이 부실하면(엉뚱한 데이터, 누락된 데이터) 아무리 정제·분석을 잘해도 결론이 틀립니다. "Garbage In, Garbage Out(쓰레기를 넣으면 쓰레기가 나온다)"의 가장 첫 관문이 바로 수집입니다.

## 자세히 알아보기 — 데이터를 구하는 세 가지 길

| 방법 | 설명 | 언제 쓰나 | 난이도 |
| --- | --- | --- | --- |
| 파일 내려받기 | CSV·Excel·공공데이터 포털 | 이미 정리된 데이터가 공개돼 있을 때 | 쉬움 |
| **API 호출** | 사이트가 공식 제공하는 데이터 창구 | **공식 API가 있을 때 (1순위 권장)** | 중간 |
| **웹 스크래핑** | 웹 페이지의 HTML에서 직접 추출 | API가 없고, 화면에만 데이터가 있을 때 | 중간~높음 |

> 💡 **핵심 — 스크래핑은 '마지막 수단'에 가깝습니다:** 공식 API가 있으면 **API를 먼저** 씁니다. API는 사이트가 "이렇게 가져가세요"라고 허락한 정식 통로라 안정적이고 합법적입니다. 스크래핑은 API가 없을 때 화면(HTML)에서 직접 긁는 방법이라, 사이트 구조가 바뀌면 깨지고 윤리·법 이슈도 따라옵니다(Part 8에서 다룹니다).

오늘 배우는 **웹 스크래핑(web scraping)** 은 "웹 페이지를 가져와서(요청), 그 HTML에서 원하는 값을 골라 꺼내는(파싱·추출)" 기술입니다. 이를 위해 먼저 "웹 페이지를 가져온다"가 정확히 무슨 일인지를 알아야 합니다 — 바로 HTTP입니다.

> 📌 **다른 산업에서는?** 수집은 금융에서 공시·환율 데이터를 모으고, 마케팅에서 경쟁사 가격·리뷰를 추적하고, 부동산에서 매물 시세를 집계하고, 연구에서 논문 메타데이터를 모을 때 쓰입니다. "분석할 데이터를 직접 만들어내는" 역량입니다.

### ✅ 짚고 넘어가기

다음 질문에 답할 수 있으면 다음 Part로 넘어가세요. 틀려도 괜찮습니다.

1. 데이터를 구하는 세 가지 방법은 무엇인가요?
2. 공식 API가 있는데도 스크래핑을 먼저 고려하면 안 되는 이유는?
3. "수집은 분석의 입구"라는 말은 무슨 뜻일까요?

> 💡 **다음 Part 예고:** "웹 페이지를 가져온다"는 건 사실 **내 컴퓨터가 서버에게 말을 거는** 일입니다. 이 대화의 규칙이 HTTP입니다. 스크래핑을 제대로 하려면 이 대화 구조부터 알아야 합니다.

In [ ]:
# ─────────────────────────────────────────────
# [도식] 데이터 분석 파이프라인에서 '수집'의 위치
# ─────────────────────────────────────────────
import matplotlib.pyplot as plt

if platform.system() == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
elif platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
else:
    plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.unicode_minus"] = False

fig, ax = plt.subplots(figsize=(11, 2.6))
ax.axis("off")
steps = [("수집\n(오늘)", "#42a5f5"), ("정제\n(지난 모듈)", "#cfd8dc"),
         ("분석/통계\n(다음 모듈)", "#cfd8dc"), ("전달\n(보고서)", "#cfd8dc")]
for i, (label, color) in enumerate(steps):
    x = 0.04 + i * 0.24
    ax.add_patch(plt.Rectangle((x, 0.3), 0.18, 0.4, facecolor=color, edgecolor="#37474f", lw=1.3))
    ax.text(x + 0.09, 0.5, label, ha="center", va="center", fontsize=11)
    if i < 3:
        ax.annotate("", xy=(x + 0.235, 0.5), xytext=(x + 0.18, 0.5),
                    arrowprops=dict(arrowstyle="->", lw=1.8))
ax.text(0.13, 0.12, "입구가 막히면 그 뒤가 전부 멈춥니다", ha="center", fontsize=9, color="#1565c0")
ax.set_xlim(0, 1); ax.set_ylim(0, 0.85)
plt.title("데이터 분석 파이프라인 — 수집은 '입구'", fontsize=12)
plt.show()

## HTTP 요청과 응답 — 웹은 어떻게 대화하는가

# 2. HTTP 요청과 응답 — 웹은 어떻게 대화하는가

브라우저 주소창에 `books.toscrape.com`을 치고 Enter를 누르면 페이지가 뜹니다. 이 순간 보이지 않는 곳에서 **대화 한 번**이 오갑니다. 내 컴퓨터(클라이언트)가 서버에게 "이 페이지 주세요"라고 **요청(request)** 하고, 서버가 "여기 있습니다"라며 **응답(response)** 을 돌려주죠. 이 대화의 규칙이 **HTTP**(HyperText Transfer Protocol)입니다.

스크래핑은 결국 **브라우저가 하던 이 요청을 코드로 대신 보내는 것**입니다. 그래서 요청·응답의 구조를 알아야 합니다.

> ❓ **이 파트에서 답할 질문:** 내 코드가 웹 서버에게 "데이터 주세요"라고 말하면, 무엇을 보내고 무엇을 돌려받을까요?

## 💡 쉽게 말하면 — 식당에서 주문하기

HTTP 대화는 식당 주문과 똑같습니다.

```text
   [나(클라이언트)]                         [서버]
        │   ── 요청(Request) ──►              │
        │   "메뉴 3번 주세요"(GET /menu/3)      │   주방에서 준비
        │                                     │
        │   ◄── 응답(Response) ──              │
        │   "여기요"(200 OK) + 음식(HTML)       │
        ▼                                     ▼
```

- **요청**에는 "무엇을(URL)", "어떻게(method)", "추가 정보(headers)"가 담깁니다.
- **응답**에는 "잘됐는지(상태 코드)"와 "내용물(body = HTML)"이 담깁니다.

## 자세히 알아보기

**요청(Request)의 구성**

| 요소 | 뜻 | 예 |
| --- | --- | --- |
| Method | 무엇을 하려는가 | `GET`(가져오기), `POST`(보내기) |
| URL | 어디에 | `https://books.toscrape.com/` |
| Headers | 부가 정보 | `User-Agent`(누가 요청하는가) |

**응답(Response)의 상태 코드 — 첫 글자만 봐도 됩니다**

| 코드 | 뜻 | 의미 |
| --- | --- | --- |
| `2xx` | 성공 | `200 OK` — 잘 받았다 |
| `3xx` | 이동 | `301` — 주소가 바뀌었다 |
| `4xx` | 내 잘못 | `404` 없는 페이지, `403` 접근 거부 |
| `5xx` | 서버 잘못 | `500` 서버 오류 |

실제로 요청을 한 번 보내봅시다. 연습용으로 만들어진 `httpbin.org`(요청 구조를 그대로 비춰주는 사이트)에 GET 요청을 보냅니다.

> **읽는 법:** `requests.get(url)` 한 줄이 브라우저가 하던 요청을 대신 보냅니다. 돌아온 `resp`에서 `status_code`(성공 여부)와 `headers`(부가 정보)를 확인할 수 있죠. 네트워크가 없는 환경이면 위 셀은 **로컬 폴백**으로 같은 구조를 보여줍니다 — 학습에는 지장이 없습니다.
>
> 핵심은 이겁니다: **요청을 보내면 → 상태 코드로 성공을 확인하고 → 본문(HTML)을 받는다.** 그런데 그 본문(HTML)에서 원하는 값을 어떻게 골라낼까요? 그 전에, **그 HTML 안에 정말 데이터가 들어 있는지**부터 따져야 합니다. 이것이 정적·동적 페이지 이야기입니다.

> ⚠ **주의 — User-Agent와 timeout:** 요청에 `User-Agent`로 "나는 누구"인지 밝히고, `timeout`으로 "몇 초까지만 기다리겠다"를 정하는 건 예의이자 안전장치입니다. timeout이 없으면 서버가 응답을 안 줄 때 코드가 영원히 멈출 수 있어요.

### 스스로 해보자! ✏️ (1)

> 정답은 하나가 아닙니다. 일단 실행해보세요.

상태 코드의 의미를 정리해보세요. 아래 딕셔너리에 `403`과 `500`의 뜻을 채워 넣고, 임의의 코드를 받았을 때 "성공/실패"를 출력하는 코드를 완성해보세요.

<details>
<summary>💡 힌트 (클릭)</summary>

```python
status_meaning = {200: "성공 — 잘 받았다", 404: "없는 페이지",
                  403: "접근 거부", 500: "서버 오류"}

def describe_status(code):
    kind = "성공" if 200 <= code < 300 else "실패/확인 필요"
    print(f"{code}: {status_meaning.get(code, '기타')} ({kind})")

describe_status(200)
describe_status(403)
describe_status(500)
```

</details>

### ✅ 짚고 넘어가기

1. 요청(Request)에 담기는 세 가지(Method·URL·Headers)는 각각 무엇을 정하나요?
2. 상태 코드 `404`와 `403`은 각각 무슨 뜻인가요?
3. `timeout`을 설정하는 이유는 무엇일까요?

> 💡 **다음 Part 예고:** 응답으로 HTML을 받았다고 해서 그 안에 항상 데이터가 들어 있는 건 아닙니다. **화면엔 보이는데 HTML 소스엔 없는** 경우가 있어요. 왜 그럴까요? 정적 페이지와 동적 페이지의 차이를 알아봅시다.

In [ ]:
# [네트워크 필요] — 연결이 안 되면 자동으로 로컬 폴백으로 같은 구조를 학습합니다.
# 연습 전용 사이트 httpbin.org 에 GET 요청을 보내, 응답 구조를 살펴봅니다.
url = "https://httpbin.org/get"
headers = {"User-Agent": "modulabs-edu-bot/1.0 (학습용)"}  # 내가 누구인지 예의 있게 밝힙니다.

try:
    if not HAS_REQUESTS:
        raise RuntimeError("requests 미설치")
    resp = requests.get(url, headers=headers, timeout=5)
    status = resp.status_code
    ctype = resp.headers.get("Content-Type", "?")
    source = "실제 네트워크 응답"
except Exception as e:
    # 로컬 폴백: 미리 저장해 둔 응답 구조로 동일하게 학습합니다.
    status = 200
    ctype = "application/json"
    source = f"로컬 폴백 (네트워크 불가: {type(e).__name__})"

print("출처     :", source)
print("상태 코드 :", status, "→", "성공(2xx)" if 200 <= status < 300 else "확인 필요")
print("Content-Type:", ctype, "  ← 응답 본문이 어떤 형식인지 알려줍니다")
print("\n해석: 200이면 서버가 요청을 정상 처리한 것입니다. 이제 응답 본문(body)에서 데이터를 꺼내면 됩니다.")

In [ ]:
status_meaning = {200: "성공 — 잘 받았다", 404: "없는 페이지",
                  403: "접근 거부", 500: "서버 오류"}

def describe_status(code):
    kind = "성공" if 200 <= code < 300 else "실패/확인 필요"
    print(f"{code}: {status_meaning.get(code, '기타')} ({kind})")

describe_status(200)
describe_status(403)
describe_status(500)

## 정적 vs 동적 페이지 — 소스에 데이터가 있는가

# 3. 정적 vs 동적 페이지 — 소스에 데이터가 있는가

스크래핑 초보가 가장 많이 부딪히는 벽이 있습니다. 분명히 브라우저 화면에는 상품 목록이 잔뜩 보이는데, 코드로 HTML을 받아 보면 **그 데이터가 텅 비어 있는** 경우입니다. "분명 보이는데 왜 못 긁지?" — 이 미스터리의 답이 **정적 vs 동적**입니다.

> ❓ **이 파트에서 답할 질문:** 서버가 보내준 HTML과, 브라우저 화면에 보이는 내용이 다를 수 있을까요?

## 💡 쉽게 말하면 — 인쇄된 책 vs 팝업북

```text
[정적 페이지 = 인쇄된 책]            [동적 페이지 = 팝업북]
 서버가 '완성된' HTML을 보냄           서버는 '뼈대'만 보냄 (내용 비어 있음)
 펼치면 글이 그대로 다 있음            펼친 뒤 JavaScript가 그림을 '세워' 채움
 → requests로 받은 소스에             → requests로 받은 소스엔 빈 뼈대뿐
   데이터가 그대로 있음                 (JS가 실행돼야 데이터가 나타남)
```

- **정적(static):** 서버가 데이터까지 채워진 완성 HTML을 보냅니다. `requests`로 받으면 데이터가 그대로 들어 있죠. → **BeautifulSoup으로 바로 추출 가능.**
- **동적(dynamic):** 서버는 빈 뼈대만 보내고, 브라우저에서 **JavaScript가 실행되며** 데이터를 채웁니다. `requests`는 JS를 실행하지 않으므로 빈 뼈대만 받습니다. → **BeautifulSoup만으론 안 됨**(다음 시간 Playwright가 필요).

## 자세히 알아보기 — 어떻게 판별하나

| 신호 | 정적일 가능성 | 동적일 가능성 |
| --- | --- | --- |
| 브라우저 "페이지 소스 보기"에 데이터가 있음 | ✅ | |
| 소스엔 없는데 화면엔 보임 | | ✅ |
| `requests`로 받은 HTML에 값이 들어 있음 | ✅ | |
| 무한 스크롤·"더 보기" 클릭으로 내용이 늘어남 | | ✅ |

직접 두 종류의 HTML을 만들어 차이를 눈으로 확인해봅시다. (둘 다 노트북 안에 넣어 둔 문자열이라 인터넷이 필요 없습니다.)

> **읽는 법:** 같은 "상품 목록 페이지"인데, 정적 HTML에서는 상품명 2개가 깔끔히 추출되고, 동적 HTML에서는 카드가 **0개** 나옵니다. 데이터가 `<script>` 안에만 있고 화면용 영역(`#app`)은 비어 있기 때문이죠. **이것이 "화면엔 보이는데 못 긁는" 현상의 정체입니다.**
>
> 오늘(D+010)은 **정적 페이지**에 집중합니다. 동적 페이지를 긁는 법은 다음 시간에 배웁니다. 하지만 "막히는 순간"을 지금 한 번 본 것이 중요합니다 — 다음 시간에 왜 새 도구가 필요한지 이해하게 되니까요.

> 📌 **다른 산업에서는?** 뉴스·블로그·정부 공시처럼 내용이 고정된 페이지는 대개 정적입니다. 반면 쇼핑몰의 실시간 가격, SNS 피드, 지도 검색 결과처럼 사용자 동작에 따라 바뀌는 화면은 대개 동적입니다. **수집 전에 유형부터 판별**하는 것이 도구 선택의 첫걸음입니다.

### ✅ 짚고 넘어가기

1. 정적 페이지와 동적 페이지의 가장 큰 차이는 무엇인가요?
2. `requests`로 받은 HTML이 비어 있다면 그 페이지는 정적일까요 동적일까요?
3. 동적 페이지를 BeautifulSoup만으로 긁기 어려운 이유는?

> 💡 **다음 Part 예고:** 정적 페이지에서 데이터를 꺼내려면 먼저 HTML이 **어떻게 생겼는지** 읽을 줄 알아야 합니다. HTML의 뼈대, 즉 DOM 트리를 알아봅시다.

In [ ]:
# 예제: '정적' HTML — 데이터가 소스 안에 그대로 들어 있습니다.
html_static = '''
<html><body>
  <div id="product-list">
    <div class="product-card"><span class="name">무선 이어폰</span><span class="price">89,900원</span></div>
    <div class="product-card"><span class="name">코튼 셔츠</span><span class="price">29,900원</span></div>
  </div>
</body></html>
'''

soup_static = BeautifulSoup(html_static, "html.parser")
names = [tag.get_text() for tag in soup_static.select(".product-card .name")]
print("정적 페이지에서 추출한 상품명:", names)
print("→ 데이터가 소스에 있으니 BeautifulSoup으로 바로 추출됩니다.")

In [ ]:
# 예제: '동적' HTML — 데이터가 <script> 안에만 있고, 화면용 영역은 비어 있습니다.
# 실제 브라우저라면 JavaScript가 이 데이터를 #app 안에 그려넣지만,
# requests/BeautifulSoup은 JS를 실행하지 않으므로 '빈 뼈대'만 봅니다.
html_dynamic = '''
<html><body>
  <div id="app"><!-- 상품이 여기에 표시됩니다 (JavaScript가 채웁니다) --></div>
  <script>
    const products = [
      {name: "노트북", price: 1290000},
      {name: "키보드", price: 49000}
    ];
    // 브라우저에서는 위 데이터를 #app 안에 렌더링(rendering)합니다.
  </script>
</body></html>
'''

soup_dynamic = BeautifulSoup(html_dynamic, "html.parser")
app_content = soup_dynamic.select_one("#app").get_text(strip=True)
cards = soup_dynamic.select(".product-card")

print("#app 영역의 내용:", repr(app_content), "  ← 비어 있습니다!")
print("찾은 상품 카드 수:", len(cards), "개  ← 0개! 데이터가 화면용 HTML에 없습니다.")
print("\n해석: 화면엔 노트북·키보드가 보이겠지만, requests가 받는 소스엔 데이터가 없습니다.")
print("→ 이런 동적 페이지는 BeautifulSoup만으론 못 긁습니다. (다음 시간 Playwright의 등장 이유)")

## HTML과 DOM 트리 — 페이지의 뼈대 읽기

# 4. HTML과 DOM 트리 — 페이지의 뼈대 읽기

HTML에서 "상품명만", "가격만" 골라 꺼내려면 HTML이 어떤 구조인지 알아야 합니다. HTML은 사실 **상자 안에 상자가 들어 있는 계층 구조**입니다. 이 구조를 **DOM**(Document Object Model, 문서 객체 모델) 트리라고 부릅니다.

> ❓ **이 파트에서 답할 질문:** HTML은 어떤 구조로 되어 있고, 그 안에서 원하는 부분을 어떻게 가리킬까요?

## 💡 쉽게 말하면 — 폴더 안의 폴더, 가족 트리

```text
html                         태그(tag): <html>, <div> 처럼 꺾쇠로 감싼 요소
 └─ body                     계층(중첩): 태그 안에 태그가 들어감 (부모-자식)
     └─ div  (id="list")     속성(attribute): class="...", id="..." 같은 꼬리표
         ├─ div (class="card")
         │    ├─ span (class="name")  → "무선 이어폰"   ← 내용(text)
         │    └─ span (class="price") → "89,900원"
         └─ div (class="card")
              ├─ span (class="name")  → "코튼 셔츠"
              └─ span (class="price") → "29,900원"
```

원하는 값을 꺼내려면 "어느 상자(태그)의, 어떤 꼬리표(class/id)를 가진 것"인지 **주소를 짚어주면** 됩니다. 그 주소를 **선택자(selector)** 라고 합니다.

## 자세히 알아보기 — 선택자와 추출 메서드

| 개념 | 표기 | 예 |
| --- | --- | --- |
| 태그 | 이름 그대로 | `div`, `span`, `a` |
| 클래스(class) 선택 | 앞에 `.` | `.price` → class가 price인 것 |
| 아이디(id) 선택 | 앞에 `#` | `#product-list` → id가 product-list인 것 |
| 계층(자손) 선택 | 공백으로 이어 | `.card .name` → card 안의 name |

| BeautifulSoup 메서드 | 하는 일 |
| --- | --- |
| `soup.select(선택자)` | 선택자에 맞는 것을 **전부** 리스트로 (CSS 선택자) |
| `soup.select_one(선택자)` | 맞는 것 중 **첫 번째** 하나 |
| `soup.find_all("태그")` | 해당 태그를 전부 |
| `.get_text()` | 태그 안의 글자만 꺼내기 |
| `tag["속성"]` 또는 `tag.get("속성")` | 속성값 꺼내기 (예: 링크의 `href`) |

작은 HTML로 직접 다뤄봅시다.

> **읽는 법:** `select()`는 **항상 리스트**를 돌려줍니다(없으면 빈 리스트 `[]`). `select_one()`은 **하나**만, 없으면 `None`을 돌려줍니다. 초보가 가장 자주 만나는 오류가 `None.get_text()`로 인한 `AttributeError`예요. **"선택자가 맞았는지"를 항상 개수(`len`)나 `None` 여부로 확인하는 습관**이 곧 다음에 배울 'AI 코드 검증'의 핵심이 됩니다.

### 스스로 해보자! ✏️ (2)

> 정답은 하나가 아닙니다. 일단 실행해보세요.

아래 `html_mini`에서 **가격(`.price`)** 의 텍스트와, `<a>` 태그의 **링크(href)** 를 각각 꺼내 출력해보세요.

<details>
<summary>💡 힌트 (클릭)</summary>

```python
price = soup.select_one(".price").get_text()
href = soup.select_one("a.link").get("href")
print("가격:", price)
print("링크:", href)
```

</details>

### ✅ 짚고 넘어가기

1. `.price`와 `#product-list`는 각각 무엇을 가리키는 선택자인가요?
2. `select()`와 `select_one()`의 반환값 차이는?
3. `None.get_text()` 오류는 왜 생기고, 어떻게 예방할까요?

> 💡 **다음 Part 예고:** 이제 선택자를 다룰 줄 압니다. 진짜 페이지 — 상품이 여러 개 들어 있는 **모두마켓 상품 목록**에서 모든 상품을 한 번에 긁어내 봅시다.

In [ ]:
# 예제: 작은 HTML을 파싱해 선택자로 값 꺼내기
html_mini = '''
<div id="product-list">
  <div class="product-card" data-pid="P001">
    <h3 class="name">무선 이어폰</h3>
    <a class="link" href="/products/P001">상세보기</a>
    <span class="price">89,900원</span>
  </div>
</div>
'''
soup = BeautifulSoup(html_mini, "html.parser")

# 1) select_one — 맞는 첫 번째 하나
name = soup.select_one(".name").get_text()
print("상품명(.name):", name)

# 2) 속성값 꺼내기 — href, data-pid
link = soup.select_one("a.link")
print("링크(href)   :", link.get("href"))
print("상품ID(data-pid):", soup.select_one(".product-card").get("data-pid"))

In [ ]:
# 예제: find_all vs select, 그리고 자주 나는 오류 읽는 법
html_two = '''
<ul><li class="item">사과</li><li class="item">바나나</li><li class="item">체리</li></ul>
'''
soup2 = BeautifulSoup(html_two, "html.parser")

# 모두 꺼내기 — select(여럿)는 항상 '리스트'를 돌려줍니다.
items = soup2.select("li.item")
print("찾은 개수:", len(items))
print("값들:", [i.get_text() for i in items])

# 흔한 오류: 없는 선택자를 select_one 하면 None이 나오고, None에 .get_text()를 부르면 에러납니다.
ghost = soup2.select_one(".not-exist")
print("\n없는 선택자 결과:", ghost, "  ← None 입니다")
if ghost is None:
    print("→ None일 때 .get_text()를 부르면 AttributeError가 납니다. 항상 None인지 먼저 확인하세요.")

In [ ]:
# 스스로 해보자! (2)
# 위 셀에서 만든 soup(=html_mini를 파싱한 것)를 그대로 사용하세요.
soup = BeautifulSoup(html_mini, "html.parser")

# 1) 가격 텍스트를 꺼내 출력해 보세요.
price = soup.select_one(".price").get_text()
print("가격:", price)

# 2) 상세보기 링크의 href를 꺼내 출력해 보세요.
href = soup.select_one("a.link").get("href")
print("링크:", href)

## BeautifulSoup으로 정적 수집 — 모두마켓 상품 목록

# 5. BeautifulSoup으로 정적 수집 — 모두마켓 상품 목록

이제 실전입니다. 우리의 가상 쇼핑몰 **모두마켓**의 "상품 목록 페이지"를 노트북 안에 HTML로 넣어 두었습니다(인터넷 불필요). 이 페이지에서 **모든 상품의 정보를 한 번에** 긁어내 봅니다. 실제 페이지처럼 **약간 지저분하게**(가격에 공백·통화기호 혼재, 일부 값 결측) 만들어 두었습니다 — 현장의 HTML이 딱 이렇기 때문입니다.

> ❓ **이 파트에서 답할 질문:** 여러 개의 항목이 반복되는 HTML에서, 각 항목의 값들을 어떻게 한꺼번에 꺼낼까요?

## 💡 쉽게 말하면 — 반복되는 카드에서 같은 자리 꺼내기

상품 목록은 보통 **같은 모양의 카드가 반복**됩니다. 그러니 전략은 단순합니다.

```text
1) 카드 전체를 리스트로 모은다      cards = soup.select(".product-card")
2) 카드 하나하나를 돌면서           for card in cards:
3) 그 안에서 같은 자리를 꺼낸다          name  = card.select_one(".name")
                                        price = card.select_one(".price")
4) 딕셔너리로 모은다                  → [{"name":..., "price":...}, ...]
```

## 자세히 알아보기

핵심은 **"바깥에서 카드를 모으고, 안에서 값을 꺼낸다"** 는 2단계입니다. `card.select_one(...)`처럼 **카드 범위 안에서** 다시 선택하는 것이 포인트예요(전체 `soup`이 아니라 각 `card` 기준).

> **읽는 법:** 6개 카드에서 각각 5개 값을 꺼내 딕셔너리 리스트로 모았습니다. 그런데 출력을 보면 가격이 `'89,900원'`, `' 29,900 원 '`, `'₩129,900'`, `''`(빈 값)처럼 **제각각**입니다. 카테고리에도 앞뒤 공백이 섞여 있죠. **수집은 여기까지** — 이 '날것'을 분석 가능한 형태로 다듬는 것이 다음 Part(랭글링 재적용)입니다.

> 💡 **개념 연결:** 방금 한 일은 사실 HTML이라는 비정형(반쯤 정형) 데이터를 **표 형태로 바꾼 것**입니다. 다음 단계에서 지난 모듈의 문자열·결측 정제 기술을 그대로 다시 씁니다. **수집과 정제가 하나의 흐름**이라는 걸 곧 체감하게 됩니다.

### 스스로 해보자! ✏️ (3)

> 정답은 하나가 아닙니다. 일단 실행해보세요.

위 `products_raw`에서 **별점(`rating`)이 4.5 이상인 상품명**만 골라 출력해보세요. (힌트: rating이 빈 문자열인 상품은 건너뛰어야 합니다.)

<details>
<summary>💡 힌트 (클릭)</summary>

```python
high_rated = []
for row in products_raw:
    r = row["rating"].strip()
    if r != "" and float(r) >= 4.5:
        high_rated.append(row["name"])
print("별점 4.5 이상:", high_rated)
```

빈 문자열을 `float()`로 바꾸려 하면 `ValueError`가 납니다. 그래서 `if r != ""`로 먼저 걸러줍니다 — 이것도 일종의 결측 처리입니다.

</details>

### ✅ 짚고 넘어가기

1. 여러 항목을 추출할 때 "바깥에서 모으고, 안에서 꺼낸다"는 무슨 뜻인가요?
2. `card.select_one(...)`과 `soup.select_one(...)`은 어떻게 다른가요?
3. 추출 직후의 데이터가 '날것'이라는 것은 어떤 의미인가요?

> 💡 **다음 Part 예고:** 긁어낸 데이터는 가격·카테고리가 지저분합니다. 지난 모듈에서 배운 **문자열·결측 정제**를 그대로 다시 적용해, 분석 가능한 깔끔한 `products` 표로 만들어봅시다.

In [ ]:
# ─────────────────────────────────────────────
# 모두마켓 상품 목록 페이지 (노트북 내장 HTML — 인터넷 불필요, 항상 실행됩니다)
# 실제 현장처럼 '오염'을 일부러 심어 둡니다: 가격에 공백/통화기호 혼재, 일부 값 결측.
# ─────────────────────────────────────────────
modumarket_html = '''
<html><head><title>모두마켓 - 상품 목록</title></head>
<body>
  <h1>오늘의 추천 상품</h1>
  <div id="product-list">
    <div class="product-card" data-pid="P001">
      <h3 class="name">무선 이어폰</h3>
      <span class="category">가전</span>
      <span class="price">89,900원</span>
      <span class="rating">4.5</span>
    </div>
    <div class="product-card" data-pid="P002">
      <h3 class="name">코튼 셔츠</h3>
      <span class="category">패션</span>
      <span class="price"> 29,900 원 </span>
      <span class="rating">4.2</span>
    </div>
    <div class="product-card" data-pid="P003">
      <h3 class="name">수분 크림</h3>
      <span class="category"> 뷰티 </span>
      <span class="price">19,900원</span>
      <span class="rating">4.7</span>
    </div>
    <div class="product-card" data-pid="P004">
      <h3 class="name">유기농 견과 세트</h3>
      <span class="category">식품</span>
      <span class="price"></span>
      <span class="rating">4.0</span>
    </div>
    <div class="product-card" data-pid="P005">
      <h3 class="name">베스트셀러 도서</h3>
      <span class="category">도서</span>
      <span class="price">12,000원</span>
      <span class="rating"></span>
    </div>
    <div class="product-card" data-pid="P006">
      <h3 class="name">블루투스 스피커</h3>
      <span class="category">가전</span>
      <span class="price">₩129,900</span>
      <span class="rating">4.3</span>
    </div>
  </div>
</body></html>
'''
soup = BeautifulSoup(modumarket_html, "html.parser")

print("페이지 제목:", soup.select_one("title").get_text())
cards = soup.select(".product-card")
print("발견한 상품 카드 수:", len(cards), "개")

In [ ]:
# 예제: 모든 상품을 한 번에 추출 — '바깥에서 카드 모으고, 안에서 값 꺼내기'
products_raw = []
for card in cards:                                   # 1) 카드 하나씩 돌며
    pid = card.get("data-pid")                        #    속성에서 상품ID
    name = card.select_one(".name").get_text()        #    카드 안에서 이름
    category = card.select_one(".category").get_text()
    price = card.select_one(".price").get_text()      #    (지저분한 채로 일단 그대로)
    rating = card.select_one(".rating").get_text()
    products_raw.append({                             # 2) 딕셔너리로 모으기
        "product_id": pid, "name": name,
        "category": category, "price": price, "rating": rating,
    })

print(f"추출한 상품 {len(products_raw)}개. 앞 3개를 봅니다 (아직 '날것' 상태):")
for row in products_raw[:3]:
    print(row)

In [ ]:
# 스스로 해보자! (3)
# products_raw를 돌면서, rating이 비어있지 않고 4.5 이상인 상품의 name을 모아 보세요.
high_rated = []
for row in products_raw:
    r = row["rating"].strip()
    if r != "" and float(r) >= 4.5:
        high_rated.append(row["name"])
print("별점 4.5 이상:", high_rated)

## 수집 → DataFrame → 랭글링 재적용

# 6. 수집 → DataFrame → 랭글링 재적용

수집은 끝이 아니라 **시작**입니다. 방금 긁은 `products_raw`는 가격이 `'89,900원'`, `'₩129,900'`, `''`처럼 제각각이라 그대로는 합계도 평균도 낼 수 없습니다. 여기서 지난 모듈의 기술이 빛을 발합니다 — **문자열 정제, 타입 변환, 결측 처리.** 수집한 데이터에 랭글링을 다시 적용해 분석 가능한 표로 만듭니다.

> ❓ **이 파트에서 답할 질문:** 웹에서 긁어온 '날것' 데이터를, 어떻게 분석 가능한 깔끔한 표로 바꿀까요?

## 자세히 알아보기 — 수집 후 정제 3단계

```text
1) DataFrame으로 변환     pd.DataFrame(products_raw)
2) 문자열 정제             가격에서 숫자만 남기기 (원, ₩, 쉼표, 공백 제거)
                          카테고리 앞뒤 공백 제거 (.str.strip())
3) 타입 변환 + 결측 처리    문자 → 숫자(pd.to_numeric), 빈 값 → NaN → 판단
```

## 데이터로 확인해 봅시다

> **읽는 법:** 처음의 `'₩129,900'`, `' 29,900 원 '` 같던 가격이 깔끔한 숫자가 되었고, 결측은 의도적으로 판단해 처리했습니다. 마지막 줄에서 **카테고리별 평균 가격**이 계산되죠 — 웹에서 긁어온 데이터로 **첫 분석**을 해낸 것입니다. 이것이 "수집은 랭글링의 입구"라는 말의 실제 모습입니다.

> 💡 **개념 연결 + 핵심 태도[판단 기록]:** 별점 결측을 '평균 대체', 가격 결측을 '집계 제외'로 **다르게** 처리했고 그 근거를 주석에 남겼습니다. 지난 모듈의 "왜 이 처리를 택했는가 기록하기"가 수집 데이터에도 똑같이 적용됩니다.

> 📌 **다른 산업에서는?** 부동산 매물 가격("3억 2,000만원"), 채용공고 연봉("4,000~5,000만원"), 리뷰 평점("★★★★☆")처럼 웹에서 긁은 값은 거의 항상 문자열 정제가 필요합니다. 수집과 정제는 늘 한 세트입니다.

### 스스로 해보자! ✏️ (4)

> 정답은 하나가 아닙니다. 일단 실행해보세요.

`products_final`에서 **가격이 있는 상품만** 골라, 가격이 비싼 순으로 정렬해 상위 3개를 출력해보세요.

<details>
<summary>💡 힌트 (클릭)</summary>

```python
top3 = (
    products_final
    .dropna(subset=["price"])
    .sort_values("price", ascending=False)
    .head(3)
)
display(top3[["name", "category", "price"]])
```

</details>

### ✅ 짚고 넘어가기

1. 긁어온 가격 문자열에서 숫자만 남기려면 어떤 정제가 필요한가요?
2. 빈 문자열을 숫자로 바꾸려 하면 어떤 일이 생기고, 어떻게 처리했나요?
3. "수집은 랭글링의 입구"라는 말을, 오늘 한 작업으로 설명할 수 있나요?

> 💡 **다음 Part 예고:** 지금까진 코드를 직접 짰지만, 현장에선 **AI에게 스크래퍼를 받아 쓰는** 일이 많습니다. 단, 그대로 믿으면 위험하죠. 도구를 어떻게 고르고, AI 코드를 어떻게 **검증**하는지 배워봅시다.

In [ ]:
# 1단계) 딕셔너리 리스트 → DataFrame
df = pd.DataFrame(products_raw)
print("수집 결과를 DataFrame으로:", df.shape)
display(df)

In [ ]:
# 2단계) 문자열 정제 — 가격에서 '숫자만' 남기기
# 정규표현식으로 숫자(0-9)가 아닌 모든 문자(원, ₩, 쉼표, 공백)를 제거합니다.
df["price_clean"] = (
    df["price"]
    .str.replace(r"[^0-9]", "", regex=True)   # 숫자 외 전부 제거
    .replace("", np.nan)                       # 빈 문자열은 결측(NaN)으로
)

# 카테고리의 앞뒤 공백도 제거 (지난 모듈의 .str.strip())
df["category"] = df["category"].str.strip()

print("가격 정제 전후 비교:")
display(df[["name", "price", "price_clean", "category"]])

In [ ]:
# 3단계) 타입 변환 + 결측 처리
# 문자 → 숫자로 변환 (결측은 NaN 유지)
df["price_clean"] = pd.to_numeric(df["price_clean"])
df["rating"] = pd.to_numeric(df["rating"].str.strip().replace("", np.nan))

print("자료형 확인 — price_clean이 숫자가 되었나?")
print(df[["price_clean", "rating"]].dtypes)

# 결측 진단 (지난 모듈의 습관: 먼저 얼마나 비었는지 본다)
print("\n[결측 진단]")
print(df[["price_clean", "rating"]].isna().sum())

# 비즈니스 판단: 가격 결측은 '집계에서 제외', 별점 결측은 '평균으로 대체'한다고 결정
df_priced = df.dropna(subset=["price_clean"]).copy()        # 가격 없는 상품은 매출 집계서 제외
df["rating"] = df["rating"].fillna(df["rating"].mean().round(1))  # 별점은 평균 대체

print("\n→ 판단 기록: 가격 결측 1건은 집계 제외, 별점 결측 1건은 평균으로 대체했다.")

In [ ]:
# 최종) 모두마켓 'products' 스키마로 정리 — 다음 모듈로 넘길 분석용 표
products_final = df[["product_id", "category", "price_clean", "rating", "name"]].rename(
    columns={"price_clean": "price"}
)
print("=== 분석 가능한 최종 products 표 ===")
display(products_final)

# 이제 집계가 됩니다! 카테고리별 평균 가격 — 수집한 데이터로 첫 분석
print("\n[카테고리별 평균 가격] — 수집 → 정제 → 분석이 이어졌습니다")
print(products_final.dropna(subset=["price"]).groupby("category")["price"].mean().round(0))

In [ ]:
# 스스로 해보자! (4)
# 1) price가 결측이 아닌 행만 남기고
# 2) price 내림차순 정렬 후 상위 3개를 출력해 보세요.
top3 = (
    products_final
    .dropna(subset=["price"])
    .sort_values("price", ascending=False)
    .head(3)
)
display(top3[["name", "category", "price"]])

## 도구 지형 + AI 보조 코딩 — 고르고, 받고, 검증하기

# 7. 도구 지형 + AI 보조 코딩 — 고르고, 받고, 검증하기

스크래핑 도구는 여러 가지입니다. 페이지 유형에 맞는 도구를 골라야 하고, 요즘은 **AI에게 스크래퍼 코드를 받아** 쓰는 경우가 많습니다. 하지만 AI가 준 코드를 **검증 없이 믿으면** 엉뚱한 데이터를 긁거나 빈 결과를 받습니다. 이 Part의 핵심 메시지는 하나입니다 — **AI는 속도를 주고, 판단과 검증은 사람이 한다.**

> ❓ **이 파트에서 답할 질문:** 어떤 도구를 언제 쓰고, AI가 만든 스크래퍼를 어떻게 신뢰할 수 있게 검증할까요?

## 자세히 알아보기 — 도구 지형과 선택 기준

| 도구 | 한마디 | JS 실행 | 속도 | 언제 |
| --- | --- | --- | --- | --- |
| `requests` + **BeautifulSoup** | 정적 HTML 파싱 | ❌ | 빠름 | **정적 페이지 (오늘)** |
| **Selenium** | 브라우저 자동화(원조) | ✅ | 느림 | 동적·구형 프로젝트 |
| **Playwright** | 브라우저 자동화(최신) | ✅ | 중간 | **동적 페이지 (다음 시간)** |
| 공식 **API** | 사이트가 준 정식 창구 | — | 빠름 | **있으면 무조건 1순위** |

**도구 선택 의사결정 (이 표를 외우세요)**

| 페이지 유형 신호 | 적합한 도구 |
| --- | --- |
| 공식 API가 있음 | **API 우선** (스크래핑 불필요) |
| HTML 소스에 데이터가 그대로 있음(정적) | `requests` + BeautifulSoup |
| 화면엔 보이는데 소스엔 없음(JS 렌더링) | Playwright |
| 로그인·무한스크롤·클릭이 필요 | Playwright |

## AI 보조 코딩 — 4단계 워크플로

AI에게 스크래퍼를 받을 때는 **반드시 이 4단계**를 거칩니다. 특히 ③ 검증이 가장 중요합니다.

```text
[1] 목적 정의   무엇을, 어떤 형태로 수집할지 한 문장으로
                "이 HTML에서 상품명·가격을 뽑아 리스트로 만들어줘"
[2] 코드 생성   명확한 프롬프트로 초안 요청 (HTML 구조 일부를 함께 제공)
[3] 검증 ★      실제로 실행 → 결과 개수·값이 맞는지 확인 (가장 중요!)
[4] 수정        틀린 선택자/구조를 사람이 고치고, 윤리 기준(Part 8) 확인
```

> 💡 **AI에게 이렇게 요청해보세요 (프롬프트 예시):**
> *"다음 HTML에서 각 `.product-card`의 상품명(`.name`)과 가격(`.price`)을 뽑아 딕셔너리 리스트로 만드는 BeautifulSoup 코드를 작성해줘. HTML 구조는 이렇다: (구조 붙여넣기)"*
> 구조를 같이 주면 AI가 셀렉터를 정확히 맞출 확률이 올라갑니다.

이제 **AI가 흔히 저지르는 실수**를 재현하고, 검증으로 잡아내 봅시다.

> **읽는 법:** AI 코드는 **에러 없이** 실행됐지만 결과가 **0개**였습니다. 이런 '조용한 실패'가 가장 위험합니다 — 에러가 안 나니 맞는 줄 알고 넘어가기 쉽거든요. **결과 개수를 내가 아는 사실과 대조하는 검증** 한 줄이 이를 잡아냅니다.

> ✅ **AI 코드 검증 체크리스트 (수집 코드를 받으면 항상 확인):**
> 1. **개수**: 추출된 항목 수가 예상과 같은가? (0개·1개면 셀렉터 의심)
> 2. **값**: 실제 값 몇 개를 눈으로 확인했는가? (엉뚱한 태그를 긁진 않았나)
> 3. **결측**: 빈 값·None이 섞여 있지 않은가?
> 4. **정적/동적**: 동적 페이지인데 requests로 긁으려 하진 않았나?
> 5. **윤리**: robots.txt·요청 속도를 확인했는가? (다음 Part)

### 스스로 해보자! ✏️ (5)

> 정답은 하나가 아닙니다. 일단 실행해보세요.

검증 체크리스트의 ①번을 함수로 만들어보세요. `verify_count(selector, expected)`: 주어진 선택자로 긁은 개수가 기대값과 같으면 "통과", 다르면 "⚠ 불일치"를 출력하는 함수입니다.

<details>
<summary>💡 힌트 (클릭)</summary>

```python
def verify_count(selector, expected):
    found = len(soup.select(selector))
    status = "통과 ✅" if found == expected else "⚠ 불일치"
    print(f"'{selector}' → {found}개 (기대 {expected}개) {status}")

verify_count(".product-card", 6)
verify_count(".product", 6)
```

</details>

### ✅ 짚고 넘어가기

1. 정적/동적/로그인 필요 — 각 상황에 맞는 도구는?
2. AI 보조 코딩 4단계 중 가장 중요한 단계와 그 이유는?
3. '조용한 실패'란 무엇이고, 어떻게 잡아내나요?

> 💡 **다음 Part 예고:** 기술은 갖췄습니다. 그런데 **"긁어도 되는가?"** 는 다른 문제입니다. 남의 서버에 요청을 보내는 일에는 책임이 따릅니다. 마지막으로 윤리·법을 배웁니다.

In [ ]:
# [AI 보조 코딩 — ③ 검증 시연]
# AI가 아래 코드를 줬다고 가정합니다. 그런데 선택자가 '.product'로 되어 있습니다.
# (실제 HTML의 클래스는 '.product-card'인데, AI가 흔히 비슷한 이름으로 틀립니다.)
ai_generated = []
for card in soup.select(".product"):          # ← AI가 준 셀렉터 (틀렸을 수 있음!)
    ai_generated.append(card.select_one(".name").get_text())

# ★ 검증: 결과 개수를 '내가 아는 사실'과 대조합니다.
print("AI 코드가 추출한 상품 수:", len(ai_generated))
print("내가 페이지에서 확인한 실제 상품 수:", len(soup.select(".product-card")))
if len(ai_generated) == 0:
    print("→ ⚠ 0개! 에러는 안 났지만 결과가 비었습니다. 선택자가 틀렸다는 신호입니다.")
    print("   (이런 '조용한 실패'를 잡으려면 반드시 개수를 검증해야 합니다.)")

In [ ]:
# [AI 보조 코딩 — ④ 수정]
# 검증에서 '0개'를 발견했으니, 실제 HTML을 보고 선택자를 '.product-card'로 고칩니다.
fixed = []
for card in soup.select(".product-card"):     # ← 사람이 검증 후 수정한 셀렉터
    fixed.append(card.select_one(".name").get_text())

print("수정 후 추출한 상품 수:", len(fixed))
print("상품명:", fixed)
print("\n→ 검증(개수 대조) → 수정(올바른 셀렉터)으로 신뢰할 수 있는 코드가 되었습니다.")
print("   AI는 속도를 줬고, 판단·검증은 사람이 했습니다.")

In [ ]:
# 스스로 해보자! (5)
def verify_count(selector, expected):
    found = len(soup.select(selector))
    status = "통과" if found == expected else "⚠ 불일치"
    print(f"'{selector}' → {found}개 (기대 {expected}개) {status}")

verify_count(".product-card", 6)
verify_count(".product", 6)

## 책임 있는 수집 — 윤리·법·안전

# 8. 책임 있는 수집 — 윤리·법·안전

> 이 Part는 코드보다 **태도**가 핵심입니다. 기술이 있다고 해서 다 해도 되는 건 아니니까요.

스크래핑은 **남의 서버에 요청을 보내는** 행위입니다. 잘못하면 서버에 부담을 주거나(과도한 요청), 허락되지 않은 데이터를 가져가거나(약관 위반), 개인정보를 침해할 수 있습니다. 그래서 **"긁기 전에 허용 범위를 먼저 판단한다"** 가 분석가의 책임이자 습관입니다.

> ❓ **이 파트에서 답할 질문:** 이 페이지를, 이런 방식으로 수집해도 괜찮은지 어떻게 판단할까요?

## 자세히 알아보기 — 수집 전 4가지 점검

| 점검 | 무엇을 | 어떻게 |
| --- | --- | --- |
| **robots.txt** | 사이트가 허용/금지한 경로 | `사이트/robots.txt` 확인, 코드로 검사 |
| **이용약관(ToS)·저작권** | 데이터 재사용·재배포 가능 여부 | 약관 읽기. 수집 가능 ≠ 재배포 가능 |
| **요청 예의(rate limit)** | 서버에 부담 안 주기 | 요청 사이 `time.sleep()`, 동시 요청 자제 |
| **개인정보** | 개인 식별 데이터 | 수집·저장 최소화, 가능하면 회피 |

먼저 `robots.txt`를 **코드로 확인**하는 법입니다. 파이썬 표준 라이브러리에 도구가 있습니다.

> **읽는 법:** `RobotFileParser`로 `can_fetch()`를 호출하면 그 경로를 긁어도 되는지 `True/False`로 알려줍니다. 코드를 짜기 **전에** 이걸 확인하는 게 책임 있는 수집의 첫걸음입니다. 여러 페이지를 돌 때는 `time.sleep()`으로 **간격을 둬서** 서버에 부담을 주지 않습니다.

> ⚠ **반드시 기억할 경계 (이 교안이 다루지 않는 것):**
> - 실제 상업 사이트(쇼핑몰·SNS 등)를 무단으로 긁는 코드는 **다루지 않습니다.** 법적·윤리적 위험 때문입니다. 연습은 스크래핑이 **허용된 샌드박스**(`books.toscrape.com` 등)에서만 합니다.
> - 로그인 우회, CAPTCHA 회피 같은 **탐지 회피 기법은 다루지 않습니다.** 그건 수집이 아니라 침입에 가깝습니다.
> - **"수집 가능"과 "재배포·상업적 이용 가능"은 다른 문제**입니다. 데이터를 긁었어도 그걸 공개·판매하려면 저작권·약관을 따로 확인해야 합니다.

> 💡 **핵심 태도[허용 범위 판단]:** 지난 모듈들에서 "왜 이 처리를 택했는가"를 기록했듯, 수집에서는 **"이 대상을 이 방식으로 긁어도 되는 근거"** 를 먼저 판단하고 기록합니다. 도구를 고르기 전에 **유형(정적/동적)과 허용 범위(robots/약관)** 부터 본다 — 이것이 이 모듈의 핵심 습관입니다.

### 스스로 해보자! ✏️ (6)

> 정답은 하나가 아닙니다. 일단 실행해보세요.

위 `robots_text`에 `Disallow: /search` 규칙이 있다고 가정하고, `/search?q=shoes` 경로를 긁어도 되는지 `can_fetch`로 확인해보세요. (robots_text를 수정해 다시 parse 해야 합니다.)

<details>
<summary>💡 힌트 (클릭)</summary>

```python
robots_text2 = '''
User-agent: *
Disallow: /private/
Disallow: /search
Allow: /
'''
rp2 = RobotFileParser()
rp2.parse(robots_text2.strip().splitlines())
print("/search?q=shoes 수집 허용?:", rp2.can_fetch("modulabs-edu-bot", "/search?q=shoes"))
# False가 나오면, 이 경로는 긁지 않아야 한다는 뜻입니다.
```

</details>

### ✅ 짚고 넘어가기

1. robots.txt는 무엇을 알려주는 파일인가요?
2. "수집 가능"과 "재배포 가능"이 다른 이유는?
3. 여러 페이지를 긁을 때 `time.sleep()`을 넣는 이유는?

> 💡 **다음 Part 예고:** 이제 정적 수집의 전 과정(요청→파싱→추출→정제→윤리)을 배웠습니다. 종합 실습에서 이 모두를 하나의 파이프라인으로 묶어봅시다.

# 🧪 종합 실습 — 모두마켓 상품 수집 파이프라인

배운 것을 **하나의 파이프라인**으로 묶을 차례입니다. 미션은 동료의 부탁 그대로입니다.

> "상품 목록 페이지에서 상품 정보를 긁어서, **분석 가능한 표**로 만들어 주세요. 가능하면 연습용 사이트에서도 같은 방식이 되는지 확인해 주고요."

```text
시나리오 1) 내장 모두마켓 HTML 수집 → 추출       (Part 5)
시나리오 2) DataFrame 변환 + 랭글링 재적용        (Part 6)
시나리오 3) 샌드박스(books.toscrape) 실제 수집     (Part 2·5, 네트워크 폴백)
마무리)     수집 리포트 양식으로 정리
```

> 이 종합 실습은 **단독 실행되도록** 데이터(HTML)를 새로 정의합니다.

## 시나리오 1 — 내장 HTML에서 상품 추출

검증 습관부터: 먼저 카드 개수를 확인하고, 그다음 추출합니다.

## 시나리오 2 — DataFrame 변환 + 랭글링 재적용

날것을 분석 가능한 표로 정제합니다.

## 시나리오 3 — 샌드박스 실제 수집 (네트워크 폴백 포함)

이번엔 **스크래핑이 허용된 연습 전용 사이트** `books.toscrape.com`에서 같은 패턴을 적용합니다. 네트워크가 안 되면 동일 구조의 로컬 HTML로 자동 대체됩니다.

> **읽는 법:** **실제 사이트든 로컬 폴백이든 똑같은 선택자**(`article.product_pod`)로 동작합니다. 핵심 학습(추출 패턴)은 네트워크와 무관하게 보장되죠. 출처가 무엇으로 찍히든, "바깥에서 모으고 안에서 꺼낸다"는 Part 5의 패턴이 그대로 재사용됩니다.

## 📊 수집 리포트로 정리하기

> 📝 **데이터 수집 리포트 (양식 예시)**
>
> **1. 수집 개요**
> - 대상: 모두마켓 상품 목록 페이지 (+ 샌드박스 books.toscrape.com에서 패턴 검증)
> - 방법: `requests`(또는 내장 HTML) → BeautifulSoup 정적 파싱 → DataFrame → 문자열·결측 정제
> - 페이지 유형 판별: **정적**(소스에 데이터 존재) → BeautifulSoup 선택
>
> **2. 수집 결과 (위 셀 숫자로 채우세요)**
> - 상품 **{n_items}개** 수집, 가격 확보 **{n_priced}개** / 결측 **{n_missing}개**.
> - 평균 가격 약 **{avg_price}원**, 최고가 카테고리는 **{top_cat}**.
>
> **3. 데이터 신뢰성 (반드시 기록)**
> - 검증: 추출 카드 수를 페이지 실제 항목 수와 대조함(조용한 실패 방지).
> - 정제: 가격 통화기호·공백 제거 후 숫자 변환, 결측 1건은 집계 제외로 판단.
>
> **4. 윤리 점검 (반드시 기록)**
> - robots.txt 확인 결과 수집 경로 허용됨. 요청 간 `time.sleep()` 적용.
> - 샌드박스(연습 허용 사이트)만 사용. 개인정보 미수집.
>
> 💡 위 `{...}` 칸을 마무리 셀 출력으로 채우면 리포트가 완성됩니다. **"왜 이 도구·이 처리·이 수집 범위였는가"의 근거를 반드시 함께 적으세요** — 이것이 이 모듈의 핵심 태도입니다.

# ✅ 오늘의 퀴즈

배운 내용을 잠깐 확인해볼게요. 틀려도 괜찮습니다.

### 개념 퀴즈

1. 내 코드가 서버에 "이 페이지 주세요"라고 보내는 것을 무엇이라 하나요? 서버가 돌려주는 것은?
2. 상태 코드 `200`과 `404`는 각각 무슨 뜻인가요?
3. "화면엔 보이는데 HTML 소스엔 데이터가 없는" 페이지는 정적일까요 동적일까요? 그런 페이지엔 어떤 도구가 필요한가요?
4. AI가 준 스크래퍼 코드가 에러 없이 실행됐는데 결과가 0개입니다. 무엇을 의심하고, 어떻게 확인하나요?
5. 수집 전에 사이트가 허용/금지한 경로를 알려주는 파일의 이름은?

### 코드 퀴즈

아래 HTML에서 **모든 상품의 이름과 가격**을 뽑아 딕셔너리 리스트로 만드세요. 모범 답안을 바로 아래에 둡니다.

> **읽는 법:** Part 5에서 배운 "바깥에서 카드 모으고, 안에서 값 꺼내기" 패턴 그대로입니다. 마지막에 **개수 검증**까지 붙이는 습관을 잊지 마세요.

# 🎓 정리 & 다음 시간 예고

## 오늘 배운 것 — 개념 연결 도식

오늘은 **웹에서 데이터를 꺼내 분석 가능한 표로 만드는** 전 과정을 배웠습니다.

```text
[Part 1] 왜 수집인가      수집은 분석의 입구 (API 우선, 스크래핑은 차선)
   ↓
[Part 2] HTTP 요청·응답   요청→상태코드 확인→본문(HTML) 받기
   ↓
[Part 3] 정적 vs 동적     소스에 데이터가 있나? (없으면 동적 → 오늘은 못 긁음)
   ↓
[Part 4] HTML / DOM       태그·속성·계층, 선택자로 주소 짚기
   ↓
[Part 5] BeautifulSoup    바깥에서 카드 모으고, 안에서 값 꺼내기
   ↓
[Part 6] 수집 → 정제      DataFrame 변환 + 문자열·결측 랭글링 재적용
   ↓
[Part 7] 도구 + AI 보조   도구 선택, AI 코드 '검증→수정' (조용한 실패 잡기)
   ↓
[Part 8] 책임 있는 수집   robots.txt·요청 예의·약관·개인정보
   ↓
[종합 실습] 수집→정제→리포트 (네트워크 폴백 포함)
```

## 한 장 정리 표

| 도구/개념 | 한마디로 | 핵심 포인트 |
| --- | --- | --- |
| HTTP | 웹의 대화 규칙 | 요청(method·URL·headers) → 응답(상태코드·body) |
| 정적/동적 | 소스에 데이터가 있나 | 정적=BS로 바로, 동적=Playwright 필요 |
| DOM·선택자 | 페이지 뼈대·주소 | `.class` `#id`, select / select_one |
| BeautifulSoup | 정적 HTML 파싱 | 바깥에서 모으고 안에서 꺼내기 |
| 수집→정제 | 랭글링 재적용 | 문자열·결측 정제로 분석 가능한 표로 |
| AI 보조 | 속도는 AI, 검증은 사람 | 개수 대조로 '조용한 실패' 잡기 |
| 윤리 | 긁기 전에 판단 | robots.txt·rate limit·약관·개인정보 |

## 페이지 유형 → 도구 선택 매핑

| 페이지 유형 신호 | 적합한 도구 | 다룬 곳 |
| --- | --- | --- |
| 공식 API가 있음 | API 우선 | 공통(1순위) |
| 소스에 데이터가 그대로 있음(정적) | requests + BeautifulSoup | **오늘** |
| 화면엔 보이는데 소스엔 없음(동적) | Playwright | 다음 시간 |
| 로그인·무한스크롤·클릭 필요 | Playwright | 다음 시간 |

## 🎓 다음 시간 예고 — D+011 동적 페이지 수집

오늘 Part 3에서 **동적 페이지**를 만났을 때를 기억하시나요? `requests`로 받은 소스엔 데이터가 없어서 BeautifulSoup이 **0개**를 돌려줬습니다. 화면엔 분명 보이는데 말이죠.

> **다음 시간:** 그 벽을 넘습니다. **JavaScript가 그려내는 화면 너머의 데이터**를 어떻게 가져올까요? 실제 브라우저를 코드로 움직이는 **Playwright**를 배우고, AI 보조 코딩 워크플로(목적→생성→검증→수정)를 본격적으로 적용하며, robots.txt와 수집 윤리를 더 깊이 다룹니다. 그리고 수집·정제가 끝나면 — 그 데이터로 *무엇을 말할 수 있는지*, 통계 모듈로 넘어갑니다.

# 📝 오늘의 과제

> **노트북 안은 짧은 점검, 본격 적용은 노트북 밖 과제로.** 오늘 배운 수집→정제→윤리 점검을 처음부터 끝까지 적용해봅니다.

## 미션: "모두마켓 신상품 수집 리포트" 만들기

아래 단계를 따라 하나의 분석 노트북을 완성하고 GitHub에 제출하세요.

1. **HTML 준비:** 종합 실습의 상품 목록 HTML을 참고해, **상품 6개 이상**(가격에 공백·통화기호 혼재, 결측 1건 포함)의 모두마켓 HTML을 직접 만드세요.
2. **유형 판별 (필수):** 이 페이지가 정적인지 동적인지 판단하고 근거를 마크다운으로 **기록**하세요.
3. **수집:** BeautifulSoup으로 모든 상품을 추출하고, **추출 개수를 검증**하세요(조용한 실패 방지).
4. **정제:** 가격을 숫자로 정제하고 결측을 판단해 처리하세요. 처리 근거를 기록하세요.
5. **분석:** 카테고리별 평균 가격, 최고가 상품 등 간단한 집계 2가지를 내세요.
6. **윤리 점검 (필수):** robots.txt 확인 방법과 요청 예의(`time.sleep`)를 리포트에 명시하세요.
7. **(심화·선택):** AI에게 스크래퍼를 요청하는 프롬프트를 작성하고, 받은 코드를 검증 체크리스트 5항목으로 점검한 과정을 적으세요.

## 제출 방법 (GitHub)

- 개인 포트폴리오 repo의 `D010/` 폴더에 `D010_웹수집_리포트.ipynb`로 저장합니다.
- 이 모듈은 3주차이므로 **작업 브랜치 → `main` PR**로 제출합니다. 강사는 PR에서 줄 단위 코드리뷰를 합니다.
- 노트북은 **처음부터 끝까지 순서대로 실행해 오류가 없어야** 합니다. (네트워크 셀은 폴백 포함)

## 평가 기준

| 축 | 무엇을 보나 |
| --- | --- |
| 정확성 | 추출·정제가 의도대로 동작하고 개수 검증을 했는가 |
| 합리성 | 페이지 유형 판별·결측 처리·도구 선택의 근거를 기록했는가 |
| 책임성 | robots.txt·요청 예의 등 윤리 점검을 수행하고 명시했는가 |

> 🚀 **더 나아가기:** 스크래핑이 허용된 샌드박스(`quotes.toscrape.com` 등)에서 같은 패턴을 적용해보세요. 또는 AI에게 일부러 모호한 프롬프트를 주고, 받은 코드의 어디가 틀리는지 검증으로 찾아보세요. 정해진 정답은 없습니다.

수고하셨습니다! 🎉

오늘 여러분은 "데이터는 주어지는 것"이 아니라 **"수집하는 것"** 임을 배웠습니다. HTTP의 원리를 이해하고, 정적 페이지에서 BeautifulSoup으로 데이터를 꺼내고, 그것을 지난 모듈의 랭글링 기술로 다듬어 분석 가능한 표로 만들었죠. 무엇보다 **AI 코드를 검증하는 눈**과 **책임 있게 수집하는 태도**를 손에 넣었습니다. 다음 시간에는 자바스크립트 너머의 데이터까지 가지러 가봅시다.

---

<sub>© 2026 모두의연구소(MODULABS). All rights reserved.<br>
제작: 교육퍼실리테이터팀 이진영 (jy.lee@modulabs.co.kr)<br>
본 교안은 생성형 AI를 활용해 제작하고 제작자가 검수했습니다.<br>
무단 복제 및 배포를 금합니다.</sub>

In [ ]:
# robots.txt 확인 — 파이썬 표준 라이브러리 urllib.robotparser
# 핵심 학습은 '규칙 해석'이므로, 규칙 내용을 직접 넣어 항상 실행되게 합니다.
from urllib.robotparser import RobotFileParser

# 예시 robots.txt 규칙 (실제 사이트들이 이런 형식을 씁니다)
robots_text = '''
User-agent: *
Disallow: /private/
Disallow: /admin/
Allow: /
Crawl-delay: 2
'''

rp = RobotFileParser()
rp.parse(robots_text.strip().splitlines())

ua = "modulabs-edu-bot"
print("이 경로를 긁어도 될까? (robots.txt 기준)")
print("  /catalogue/page-1.html :", rp.can_fetch(ua, "/catalogue/page-1.html"))  # 허용
print("  /private/secret.html   :", rp.can_fetch(ua, "/private/secret.html"))    # 금지
print("\n→ Disallow 경로는 False가 나옵니다. 코드를 짜기 전에 이렇게 먼저 확인하는 습관!")

In [ ]:
# [네트워크 필요] — 실제 사이트의 robots.txt도 같은 방식으로 읽습니다. (안 되면 위 학습으로 충분)
real_rp = RobotFileParser()
try:
    if not HAS_REQUESTS:
        raise RuntimeError("requests 미설치")
    real_rp.set_url("https://books.toscrape.com/robots.txt")
    real_rp.read()
    allowed = real_rp.can_fetch("*", "https://books.toscrape.com/catalogue/page-1.html")
    print("실제 books.toscrape.com 에서 /catalogue/page-1.html 수집 허용?:", allowed)
except Exception as e:
    print(f"네트워크 불가({type(e).__name__}) — 위 셀의 규칙 해석 학습으로 대체합니다.")
    print("실무에서는 '사이트주소/robots.txt'를 브라우저로 직접 열어 봐도 됩니다.")

In [ ]:
# 요청 예의 — 여러 페이지를 긁을 때는 요청 사이에 '쉬어 줍니다'
import time

pages = ["page-1", "page-2", "page-3"]
print("요청 예의를 지키며 순회 (각 요청 사이 1초 대기):")
for p in pages:
    # 실제로는 여기서 requests.get(...) 을 호출합니다. 지금은 예의 패턴만 시연.
    print(f"  요청 → {p}  (서버 부담을 줄이기 위해 잠시 대기)")
    time.sleep(1)   # rate limiting: 서버에 한꺼번에 몰리지 않도록
print("완료. time.sleep()으로 서버에 부담을 주지 않는 것이 기본 예의입니다.")

In [ ]:
# 스스로 해보자! (6)
robots_text2 = '''
User-agent: *
Disallow: /private/
Disallow: /search
Allow: /
'''
rp2 = RobotFileParser()
rp2.parse(robots_text2.strip().splitlines())
print("/search?q=shoes 수집 허용?:", rp2.can_fetch("modulabs-edu-bot", "/search?q=shoes"))
# False가 나오면, 이 경로는 긁지 않아야 한다는 뜻입니다.

In [ ]:
# ─────────────────────────────────────────────
# 종합 실습용 — 모두마켓 상품 목록 HTML (새 스냅샷, 단독 실행 가능)
# ─────────────────────────────────────────────
shop_html = '''
<div id="catalog">
  <div class="item" data-pid="P101"><span class="title">게이밍 마우스</span><span class="cat">가전</span><span class="cost">45,000원</span></div>
  <div class="item" data-pid="P102"><span class="title">린넨 원피스</span><span class="cat"> 패션 </span><span class="cost"> 79,000 원 </span></div>
  <div class="item" data-pid="P103"><span class="title">아로마 캔들</span><span class="cat">뷰티</span><span class="cost">₩23,000</span></div>
  <div class="item" data-pid="P104"><span class="title">프리미엄 원두</span><span class="cat">식품</span><span class="cost"></span></div>
  <div class="item" data-pid="P105"><span class="title">요가 매트</span><span class="cat">스포츠</span><span class="cost">38,500원</span></div>
</div>
'''
shop_soup = BeautifulSoup(shop_html, "html.parser")
print("종합 실습 데이터 준비 완료 — 상품 카드:", len(shop_soup.select(".item")), "개")

In [ ]:
# 시나리오 1 — 추출 (검증 → 추출)
cards = shop_soup.select(".item")
print("[검증] 추출 대상 카드 수:", len(cards), "개")

rows = []
for c in cards:
    rows.append({
        "product_id": c.get("data-pid"),
        "name": c.select_one(".title").get_text(),
        "category": c.select_one(".cat").get_text(),
        "price": c.select_one(".cost").get_text(),
    })
print("[추출] 완료. 앞 2개(날것):")
for r in rows[:2]:
    print(" ", r)

In [ ]:
# 시나리오 2 — 정제 파이프라인 (method chaining 스타일)
shop_df = pd.DataFrame(rows)

# 가격: 숫자만 남기고 → 결측은 NaN → 숫자형
shop_df["price"] = (
    shop_df["price"].str.replace(r"[^0-9]", "", regex=True).replace("", np.nan)
)
shop_df["price"] = pd.to_numeric(shop_df["price"])
# 카테고리: 앞뒤 공백 제거
shop_df["category"] = shop_df["category"].str.strip()

print("[정제 결과]")
display(shop_df)
print("\n[결측 진단] 가격 결측:", int(shop_df["price"].isna().sum()), "건  (집계 시 제외 판단)")
print("[카테고리별 평균 가격]")
print(shop_df.dropna(subset=["price"]).groupby("category")["price"].mean().round(0))

In [ ]:
# [네트워크 필요] — books.toscrape.com 은 스크래핑 연습 전용으로 공개된 샌드박스입니다.
# 네트워크가 안 되면 같은 구조의 로컬 HTML로 폴백합니다(셀렉터는 동일하게 동작).
local_books_fallback = '''
<section>
  <article class="product_pod"><h3><a title="A Light in the Attic">A Light...</a></h3><p class="price_color">£51.77</p></article>
  <article class="product_pod"><h3><a title="Tipping the Velvet">Tipping...</a></h3><p class="price_color">£53.74</p></article>
  <article class="product_pod"><h3><a title="Soumission">Soumission</a></h3><p class="price_color">£50.10</p></article>
</section>
'''

books_url = "https://books.toscrape.com/"
try:
    if not HAS_REQUESTS:
        raise RuntimeError("requests 미설치")
    resp = requests.get(books_url, timeout=5, headers={"User-Agent": "modulabs-edu-bot/1.0"})
    resp.encoding = "utf-8"   # 사이트가 UTF-8이므로 인코딩을 명시 (£ 등 기호가 깨지는 것 방지)
    html = resp.text
    source = "실제 books.toscrape.com"
except Exception as e:
    html = local_books_fallback
    source = f"로컬 폴백({type(e).__name__})"

books_soup = BeautifulSoup(html, "html.parser")
# 같은 추출 패턴: 바깥에서 product_pod 모으고, 안에서 제목·가격 꺼내기
books = []
for pod in books_soup.select("article.product_pod"):
    books.append({
        "title": pod.h3.a.get("title"),
        "price": pod.select_one("p.price_color").get_text(strip=True),
    })

print("데이터 출처:", source)
print("[검증] 추출한 책 수:", len(books), "권")
display(pd.DataFrame(books).head())

In [ ]:
# 수집 결과 요약 숫자 자동 추출
n_items = len(shop_df)
n_priced = int(shop_df["price"].notna().sum())
n_missing = int(shop_df["price"].isna().sum())
avg_price = shop_df["price"].mean()
top_cat = shop_df.dropna(subset=["price"]).groupby("category")["price"].mean().idxmax()

print("자동 추출된 수집 리포트 숫자")
print(f"  수집 상품 수   : {n_items}개")
print(f"  가격 확보      : {n_priced}개 / 결측 {n_missing}개")
print(f"  평균 가격      : {avg_price:,.0f}원")
print(f"  최고가 카테고리: {top_cat}")

In [ ]:
# 코드 퀴즈 — 모범 답안
quiz_html = '''
<div class="goods"><span class="gname">텀블러</span><span class="gprice">15,000원</span></div>
<div class="goods"><span class="gname">에코백</span><span class="gprice">9,900원</span></div>
'''
quiz_soup = BeautifulSoup(quiz_html, "html.parser")

result = []
for g in quiz_soup.select(".goods"):           # 바깥에서 카드 모으기
    result.append({
        "name": g.select_one(".gname").get_text(),    # 안에서 값 꺼내기
        "price": g.select_one(".gprice").get_text(),
    })
print("추출 결과:", result)
print("검증 — 추출 개수:", len(result), "개 (기대 2개)")